# Planners-8-Temporal — Planification Temporelle (twin C#)

Ce notebook est le **jumeau C# (.NET Interactive)** du notebook Python `Planners-8-Temporal.ipynb`. Il réimplémente **from-scratch** (BCL .NET, **0 NuGet**, 0 `unified_planning`, 0 `ortools`) les concepts de la planification temporelle : **PDDL 2.1 actions duratives**, **algèbre d'intervalles d'Allen**, **réseau temporel simple (STN)** et **ordonnancement avec contraintes de précédence et ressources**.

## Objectifs pédagogiques

1. **Modéliser une action durative** PDDL 2.1 (conditions `at start`, `over all`, `at end`, effets).
2. **Manipuler l'algèbre d'intervalles d'Allen** (13 relations binaires entre intervalles temporels).
3. **Construire et vérifier un STN** (Simple Temporal Network) par Floyd-Warshall sur la matrice des distances.
4. **Détecter les conflits temporels** (exclusion mutuelle, chevauchement interdit).
5. **Ordonnancer** des tâches avec précédence + ressources capacitaires (variante RCPSP), produire un diagramme de Gantt ASCII.

## Pourquoi un twin from-scratch ?

Le notebook Python original s'appuie sur **`unified_planning`** (API Python de planification) et **`ortools` CP-SAT (Google)** pour exprimer et résoudre le problème temporel. Ces libs ne sont pas mobilisables nativement côté .NET Interactive sans NuGet lourds. Le twin C# reconstruit les **concepts** — Allen, STN, Floyd-Warshall, scheduling — pour les rendre **transparents et exécutables** sur toute machine .NET 9+. Le *value-add* (EPIC #4956 Prong B) est pédagogique : chaque mécanisme (composition d'Allen, propagation de contraintes STN, heuristique de scheduling) est visible dans le code C#.

## Prérequis

.NET 9.0+ (kernel `csharp`). Familiarité avec la planification classique state-space (cf. `Planners-3-State-Space-Csharp.ipynb`). Aucune librairie externe.


## Setup — utilitaires d'affichage


In [1]:
using System;
using System.Collections.Generic;
using System.Linq;
using System.Text;

static void Show(object o) { try { display(o); } catch { Console.WriteLine(o); } }
static void Show(string label, object o) => Show($"{label}: {o}");

Show("Setup", "OK — BCL .NET, 0 NuGet (pas de unified_planning / ortools).");


The below script needs to be able to find the current output cell; this is an easy method to get it.

Setup: OK — BCL .NET, 0 NuGet (pas de unified_planning / ortools).

## Partie 1 — PDDL 2.1 : actions duratives

En planification classique (STRIPS/PDDL 1), une action est *instantanée*. **PDDL 2.1** (Fox & Long, 2003) introduit les **actions duratives** : une action s'étend sur un intervalle $[t_{start}, t_{end}]$ de durée $\delta$, avec des conditions et effets qualifiés temporellement :

- `at start` : condition/effect au début de l'action,
- `over all` : condition maintenue pendant toute la durée,
- `at end` : condition/effect à la fin.

Cela permet la **concurrence** : deux actions duratives peuvent se chevaucher si elles ne se violent pas mutuellement (exclusion mutuelle temporelle sur une ressource, par exemple).


In [2]:
// --- PDDL 2.1 : action durative from-scratch ---
public enum TemporalQualifier { AtStart, OverAll, AtEnd }

public sealed class TimedCondition
{
    public TemporalQualifier When { get; }
    public string Predicate { get; }   // ex: "robot_at(A)", "battery > 10"
    public TimedCondition(TemporalQualifier q, string pred) { When = q; Predicate = pred; }
    public override string ToString() => $"{When}({Predicate})";
}

public sealed class DurativeAction
{
    public string Name { get; }
    public double MinDuration { get; }
    public double MaxDuration { get; }
    public List<TimedCondition> Conditions { get; } = new();
    public List<TimedCondition> Effects { get; } = new();
    public DurativeAction(string name, double durMin, double durMax) { Name = name; MinDuration = durMin; MaxDuration = durMax; }
    public DurativeAction Cond(TemporalQualifier q, string p) { Conditions.Add(new TimedCondition(q, p)); return this; }
    public DurativeAction Eff(TemporalQualifier q, string p) { Effects.Add(new TimedCondition(q, p)); return this; }
    public override string ToString()
    {
        var sb = new StringBuilder();
        sb.AppendLine($"durative-action {Name} [duration in [{MinDuration}, {MaxDuration}]]");
        sb.AppendLine("  condition:");
        foreach (var c in Conditions) sb.AppendLine($"    {c}");
        sb.AppendLine("  effect:");
        foreach (var e in Effects) sb.AppendLine($"    {e}");
        return sb.ToString();
    }
}

// --- Action move(A, B) : deplacement classique avec duree ---
var moveAB = new DurativeAction("move(robot, A, B)", 4.0, 4.0)
    .Cond(TemporalQualifier.AtStart, "robot_at(A)")
    .Cond(TemporalQualifier.OverAll, "battery > 5")
    .Eff(TemporalQualifier.AtStart, "not robot_at(A)")
    .Eff(TemporalQualifier.AtEnd, "robot_at(B)")
    .Eff(TemporalQualifier.AtEnd, "battery -= 4");
Show("Action durative move(A,B)", moveAB.ToString());

// --- Action charge : recharge la batterie ---
var charge = new DurativeAction("charge(robot)", 3.0, 3.0)
    .Cond(TemporalQualifier.AtStart, "robot_at(dock)")
    .Eff(TemporalQualifier.AtEnd, "battery = 100");
Show("Action durative charge", charge.ToString());


Action durative move(A,B): durative-action move(robot, A, B) [duration in [4, 4]]
  condition:
    AtStart(robot_at(A))
    OverAll(battery > 5)
  effect:
    AtStart(not robot_at(A))
    AtEnd(robot_at(B))
    AtEnd(battery -= 4)


Action durative charge: durative-action charge(robot) [duration in [3, 3]]
  condition:
    AtStart(robot_at(dock))
  effect:
    AtEnd(battery = 100)


## Partie 2 — Algèbre d'intervalles d'Allen

**Allen (1983)** définit **13 relations binaires** entre deux intervalles temporels $I_1 = [s_1, e_1]$ et $I_2 = [s_2, e_2]$ (avec $s_i < e_i$) : `before`, `after`, `meets`, `met-by`, `overlaps`, `overlapped-by`, `during`, `contains`, `starts`, `started-by`, `finishes`, `finished-by`, `equals`.

Elles forment une **algèbre de relation** avec une table de composition (si $R_1(I_1,I_2)$ et $R_2(I_2,I_3)$ alors $R_1 \circ R_2 \subseteq \{\text{relations possibles}\}(I_1,I_3)$). On implémente la classification par comparaison des bornes, puis la table de composition pour la propagation de contraintes.


In [3]:
// --- Allen Interval Algebra from-scratch ---
public readonly struct Interval
{
    public double Start { get; }
    public double End { get; }
    public string Name { get; }
    public Interval(string name, double s, double e) { Name = name; Start = s; End = e; }
    public double Duration => End - Start;
    public override string ToString() => $"{Name}[{Start},{End}]";
}

public enum AllenRel
{
    Before, After, Meets, MetBy,
    Overlaps, OverlappedBy, During, Contains,
    Starts, StartedBy, Finishes, FinishedBy, Equal
}

public static class AllenAlgebra
{
    public static string RelStr(AllenRel r) => r switch
    {
        AllenRel.Before => "before", AllenRel.After => "after",
        AllenRel.Meets => "meets", AllenRel.MetBy => "met-by",
        AllenRel.Overlaps => "overlaps", AllenRel.OverlappedBy => "overlapped-by",
        AllenRel.During => "during", AllenRel.Contains => "contains",
        AllenRel.Starts => "starts", AllenRel.StartedBy => "started-by",
        AllenRel.Finishes => "finishes", AllenRel.FinishedBy => "finished-by",
        AllenRel.Equal => "equal", _ => "?"
    };

    // Classifie la relation entre deux intervalles (bornes strictement comparees).
    public static AllenRel Classify(Interval a, Interval b)
    {
        const double EPS = 1e-9;
        double s1=a.Start, e1=a.End, s2=b.Start, e2=b.End;
        bool SEq(double x, double y) => Math.Abs(x - y) < EPS;
        if (SEq(s1,s2) && SEq(e1,e2)) return AllenRel.Equal;
        if (SEq(e1,s2)) return AllenRel.Meets;
        if (SEq(s1,e2)) return AllenRel.MetBy;
        if (e1 < s2) return AllenRel.Before;
        if (s1 > e2) return AllenRel.After;
        if (SEq(s1,s2) && e1 < e2) return AllenRel.Starts;
        if (SEq(s1,s2) && e1 > e2) return AllenRel.StartedBy;
        if (SEq(e1,e2) && s1 > s2) return AllenRel.Finishes;
        if (SEq(e1,e2) && s1 < s2) return AllenRel.FinishedBy;
        if (s1 > s2 && e1 < e2) return AllenRel.During;
        if (s1 < s2 && e1 > e2) return AllenRel.Contains;
        if (s1 < s2 && e1 < e2 && e1 > s2) return AllenRel.Overlaps;
        if (s1 > s2 && e1 > e2 && s1 < e2) return AllenRel.OverlappedBy;
        return AllenRel.Equal;
    }

    // Inverse d'une relation (symetrie de l'algebre d'Allen).
    public static AllenRel Inverse(AllenRel r) => r switch
    {
        AllenRel.Before => AllenRel.After, AllenRel.After => AllenRel.Before,
        AllenRel.Meets => AllenRel.MetBy, AllenRel.MetBy => AllenRel.Meets,
        AllenRel.Overlaps => AllenRel.OverlappedBy, AllenRel.OverlappedBy => AllenRel.Overlaps,
        AllenRel.During => AllenRel.Contains, AllenRel.Contains => AllenRel.During,
        AllenRel.Starts => AllenRel.StartedBy, AllenRel.StartedBy => AllenRel.Starts,
        AllenRel.Finishes => AllenRel.FinishedBy, AllenRel.FinishedBy => AllenRel.Finishes,
        _ => AllenRel.Equal
    };
}

// --- Demonstration : 4 intervalles ---
var I1 = new Interval("I1", 0, 5);     // [0,5]
var I2 = new Interval("I2", 5, 8);     // meets I1
var I3 = new Interval("I3", 3, 7);     // overlaps I1
var I4 = new Interval("I4", 1, 4);     // during I1
var ivals = new[] { I1, I2, I3, I4 };
var sb = new StringBuilder();
sb.AppendLine("Matrice des relations d'Allen :");
sb.AppendLine($"       {string.Join("  ", ivals.Select(i => i.Name))}");
foreach (var a in ivals)
{
    sb.Append($"{a.Name}  ");
    foreach (var b in ivals)
        sb.Append($"{AllenAlgebra.RelStr(AllenAlgebra.Classify(a,b)),-12} ");
    sb.AppendLine();
}
Show("Allen", sb.ToString());
Show("Inverse de meets", AllenAlgebra.RelStr(AllenAlgebra.Inverse(AllenRel.Meets)));


Allen: Matrice des relations d'Allen :
       I1  I2  I3  I4
I1  equal        meets        overlaps     contains     
I2  met-by       equal        overlapped-by after        
I3  overlapped-by overlaps     equal        overlapped-by 
I4  during       before       overlaps     equal        


Inverse de meets: met-by

## Partie 3 — Réseau temporel simple (STN) et Floyd-Warshall

Un **Simple Temporal Network** (Dechter, Meiri & Pearl, 1991) est un graphe dont les arêtes contraignent la distance temporelle entre deux événements (timepoints). Une arête $X \xrightarrow{[a,b]} Y$ impose $a \leq t_Y - t_X \leq b$.

On encode chaque contrainte $[a,b]$ par deux arêtes orientées dans la **matrice des distances** $D$ :
- $D[X,Y] \leq b$ (borne supérieure),
- $D[Y,X] \leq -a$ (borne inférieure, retournée).

Le STN est **consistant** ssi aucun cycle négatif n'apparaît après l'exécution de **Floyd-Warshall** (all-pairs shortest path). Un cycle négatif $\Rightarrow$ contradiction temporelle insatisfiable.


In [4]:
// --- STN + Floyd-Warshall from-scratch ---
public sealed class STN
{
    public List<string> Nodes { get; } = new();
    public double[,] Dist;   // matrice des distances
    private int n => Nodes.Count;
    public STN(IEnumerable<string> nodes)
    {
        Nodes = nodes.ToList();
        int N = Nodes.Count;
        Dist = new double[N, N];
        for (int i = 0; i < N; i++)
            for (int j = 0; j < N; j++)
                Dist[i, j] = i == j ? 0 : double.PositiveInfinity;
    }
    public int Idx(string node) => Nodes.IndexOf(node);
    // Contrainte a <= tY - tX <= b
    public void AddConstraint(string x, string y, double a, double b)
    {
        int X = Idx(x), Y = Idx(y);
        Dist[X, Y] = Math.Min(Dist[X, Y], b);
        Dist[Y, X] = Math.Min(Dist[Y, X], -a);
    }
    // Floyd-Warshall : retourne true si consistant (pas de cycle negatif).
    public (bool Consistent, double[,] D) FloydWarshall()
    {
        int N = n;
        var D = (double[,])Dist.Clone();
        for (int k = 0; k < N; k++)
            for (int i = 0; i < N; i++)
                for (int j = 0; j < N; j++)
                    if (D[i, k] + D[k, j] < D[i, j])
                        D[i, j] = D[i, k] + D[k, j];
        bool consistent = true;
        for (int i = 0; i < N; i++) if (D[i, i] < 0) consistent = false;
        return (consistent, D);
    }
    public string Dump()
    {
        var sb = new StringBuilder();
        sb.AppendLine($"STN ({n} nodes): {string.Join(", ", Nodes)}");
        return sb.ToString();
    }
}

// --- STN consistant : 3 evenements A, B, C ---
var stn1 = new STN(new[] { "X0", "A", "B", "C" });
stn1.AddConstraint("X0", "A", 1, 5);    // A se produit entre t=1 et t=5
stn1.AddConstraint("X0", "B", 2, 8);    // B entre t=2 et t=8
stn1.AddConstraint("A", "B", 1, 10);    // B apres A, au moins 1 unite plus tard
stn1.AddConstraint("B", "C", 0, 4);     // C apres B, dans les 4 unites
stn1.AddConstraint("X0", "C", 0, 15);   // C avant t=15
var (ok1, D1) = stn1.FloydWarshall();
Show("STN 1", stn1.Dump());
Show("STN 1 consistant ?", ok1);
Show("Bornes X0->C", $"[{-D1[stn1.Idx("C"), stn1.Idx("X0")]}, {D1[stn1.Idx("X0"), stn1.Idx("C")]}]");

// --- STN INCONSISTANT : A avant B, B avant A (cycle negatif) ---
var stn2 = new STN(new[] { "A", "B" });
stn2.AddConstraint("A", "B", 5, 10);    // B apres A d'au moins 5
stn2.AddConstraint("B", "A", 5, 10);    // A apres B d'au moins 5 => contradiction
var (ok2, D2) = stn2.FloydWarshall();
Show("STN 2 (contradictoire)", stn2.Dump());
Show("STN 2 consistant ?", ok2);
Show("Diagonale (cycle negatif ?)", D2[0,0] < 0 ? $"OUI, D[A,A]={D2[0,0]} (contradiction)" : "non");


STN 1: STN (4 nodes): X0, A, B, C


STN 1 consistant ?: True

Bornes X0->C: [2, 12]

STN 2 (contradictoire): STN (2 nodes): A, B


STN 2 consistant ?: False

Diagonale (cycle negatif ?): OUI, D[A,A]=-10 (contradiction)

## Partie 4 — Conflits temporels et exclusion mutuelle

Deux actions duratives sont en **exclusion mutuelle temporelle** si elles ne peuvent pas se chevaucher (par exemple, deux actions utilisant exclusivement la même ressource). On détecte un conflit en testant si leurs intervalles planifiés sont en relation `overlaps` / `during` / `contains` / etc. (tout sauf `before`/`after`/`meets`/`met-by`/`equal` disjoint).


In [5]:
// --- Detection de conflits temporels ---
public static class TemporalMutex
{
    // Deux intervalles sont-ils en conflit (chevauchement) ?
    public static bool Conflicts(Interval a, Interval b)
    {
        var rel = AllenAlgebra.Classify(a, b);
        // Conflit si les intervalles se chevauchent (pas strictement ordonnes).
        return rel != AllenRel.Before && rel != AllenRel.After
            && rel != AllenRel.Meets && rel != AllenRel.MetBy;
    }
    // Verifie qu'un schedule (liste d'intervalles nommes) n'a pas de conflit 2-a-2.
    public static List<(Interval, Interval)> FindConflicts(List<Interval> schedule)
    {
        var conflicts = new List<(Interval, Interval)>();
        for (int i = 0; i < schedule.Count; i++)
            for (int j = i + 1; j < schedule.Count; j++)
                if (Conflicts(schedule[i], schedule[j]))
                    conflicts.Add((schedule[i], schedule[j]));
        return conflicts;
    }
}

// --- Schedule sans conflit ---
var sched1 = new List<Interval>
{
    new Interval("weld", 0, 4),
    new Interval("paint", 4, 7),    // meets weld, pas de conflit
    new Interval("inspect", 8, 10), // apres, pas de conflit
};
var conflicts1 = TemporalMutex.FindConflicts(sched1);
Show("Schedule 1 conflits", conflicts1.Count == 0 ? "AUCUN (OK)" : string.Join("; ", conflicts1.Select(c => $"{c.Item1.Name}<->{c.Item2.Name}")));

// --- Schedule avec conflit (meme machine) ---
var sched2 = new List<Interval>
{
    new Interval("weld_A", 0, 5),
    new Interval("weld_B", 3, 8),    // overlaps weld_A => conflit (1 machine)
};
var conflicts2 = TemporalMutex.FindConflicts(sched2);
Show("Schedule 2 conflits", conflicts2.Count == 0 ? "AUCUN" : string.Join("; ", conflicts2.Select(c => $"{c.Item1.Name}<->{c.Item2.Name} [{c.Item1}] vs [{c.Item2}]")));


Schedule 1 conflits: AUCUN (OK)

Schedule 2 conflits: weld_A<->weld_B [weld_A[0,5]] vs [weld_B[3,8]]

## Partie 5 — Ordonnancement (RCPSP-lite) et diagramme de Gantt

On assemble les briques précédentes : étant donné un ensemble de **tâches** (durée, précédences, demande en ressource) et une **capacité** de ressource, on calcule un **schedule** par heuristique gloutonne (liste de priorité + placement au plus tôt sous contraintes de précédence et de capacité). On produit ensuite un **diagramme de Gantt ASCII**.

C'est une variante simplifiée du **RCPSP** (Resource-Constrained Project Scheduling Problem, NP-difficile). L'heuristique gloutonne ne garantit pas l'optimalité (makespan minimal) mais produit un schedule réalisable en temps polynomial — la comparaison avec l'optimum known d'un benchmark (ex: PSPLIB) est laissée en exercice.


In [6]:
// --- Ordonnancement glouton RCPSP-lite + Gantt ASCII ---
public sealed class Task
{
    public string Name { get; }
    public double Duration { get; }
    public double Demand { get; }                 // demande en ressource
    public List<string> Predecessors { get; } = new();
    public double Start { get; set; } = -1;       // -1 = non planifiee
    public double Finish => Start + Duration;
    public Task(string name, double dur, double demand, params string[] preds)
    { Name = name; Duration = dur; Demand = demand; Predecessors = preds.ToList(); }
}

public sealed class Scheduler
{
    public double Capacity { get; }
    public Scheduler(double cap) { Capacity = cap; }

    // Heuristique gloutonne : a chaque pas, planifier la tache disponible
    // (predecesseurs finis) dont le placement au plus tot respecte la capacite.
    // Les evenements temporels = fins de taches deja planifiees (grille discretee).
    public void ScheduleAll(List<Task> tasks)
    {
        var done = new HashSet<string>();
        double horizon = tasks.Sum(t => t.Duration) + 1;
        // Grille de temps fine
        var grid = Enumerable.Range(0, (int)Math.Ceiling(horizon) + 1).Select(i => (double)i).ToList();
        while (done.Count < tasks.Count)
        {
            // Taches disponibles
            var ready = tasks.Where(t => !done.Contains(t.Name)
                && t.Predecessors.All(p => done.Contains(p))).ToList();
            if (ready.Count == 0) { /* deadlock cyclique */ break; }
            // Par priorite : duree la plus longue d'abord (LPT heuristic)
            ready = ready.OrderByDescending(t => t.Duration).ToList();
            bool placed = false;
            foreach (var t in ready)
            {
                // Trouver le plus tot start >= max(fins predecesseurs) tel que
                // pendant [start, start+dur] la capacite est respectee a tout instant.
                double earliest = t.Predecessors.Count == 0 ? 0
                    : t.Predecessors.Max(p => tasks.First(x => x.Name == p).Finish);
                for (double s = earliest; s <= horizon; s += 1.0)
                {
                    bool ok = true;
                    for (double tt = s; tt < s + t.Duration; tt += 1.0)
                    {
                        double used = tasks.Where(x => x.Start >= 0
                            && tt >= x.Start && tt < x.Start + x.Duration).Sum(x => x.Demand);
                        if (used + t.Demand > Capacity + 1e-9) { ok = false; break; }
                    }
                    if (ok) { t.Start = s; done.Add(t.Name); placed = true; break; }
                }
                if (placed) break;
                if (!placed && t.Start < 0) { /* impossible ce tour */ }
            }
            if (!placed) break;  // aucun placement possible => arreter (evite boucle infinie)
        }
    }

    // Diagramme de Gantt ASCII : lignes = taches, colonnes = unites de temps.
    public string Gantt(List<Task> tasks)
    {
        double makespan = tasks.Max(t => t.Finish);
        int W = (int)Math.Ceiling(makespan) + 1;
        var sb = new StringBuilder();
        sb.AppendLine($"Gantt (makespan = {makespan}, capacite = {Capacity}):");
        sb.Append("task".PadRight(12) + " |");
        for (int t = 0; t < W; t++) sb.Append((t % 10).ToString());
        sb.AppendLine();
        sb.AppendLine(new string('-', 14 + W));
        foreach (var t in tasks)
        {
            sb.Append(t.Name.PadRight(12) + " |");
            var row = new string('.', W).ToCharArray();
            for (int k = 0; k < W; k++)
                if (k >= t.Start && k < t.Start + t.Duration) row[k] = '#';
            sb.AppendLine(new string(row));
        }
        // Usage de capacite par unite de temps
        sb.Append("load".PadRight(12) + " |");
        for (int k = 0; k < W; k++)
        {
            double used = tasks.Where(x => x.Start >= 0 && k >= x.Start && k < x.Start + x.Duration).Sum(x => x.Demand);
            sb.Append(used == 0 ? "." : Math.Floor(used).ToString());
        }
        sb.AppendLine();
        return sb.ToString();
    }
}

// --- Scenario : atelier avec 2 machines (capacite = 2) ---
var tasks = new List<Task>
{
    new Task("A", 3, 1),             // duree 3, demande 1 machine
    new Task("B", 2, 1),
    new Task("C", 4, 1, "A"),        // C apres A
    new Task("D", 2, 1, "B"),
    new Task("E", 3, 2, "C", "D"),   // E apres C et D, demande 2 machines
};
var sched = new Scheduler(2);
sched.ScheduleAll(tasks);
Show("Schedule atelier", sched.Gantt(tasks));
Show("Makespan", tasks.Max(t => t.Finish));
Show("Realisable ?", tasks.All(t => t.Start >= 0) ? "OUI (toutes planifiees)" : "NON (certaines non placees)");


Schedule atelier: Gantt (makespan = 10, capacite = 2):
task         |01234567890
-------------------------
A            |###........
B            |##.........
C            |...####....
D            |..##.......
E            |.......###.
load         |2222111222.


Makespan: 10

Realisable ?: OUI (toutes planifiees)

## Conclusion

Ce twin C# a reconstruit **from-scratch** les 4 piliers de la planification temporelle :

1. **PDDL 2.1 actions duratives** — conditions `at start`/`over all`/`at end`, effets (Partie 1).
2. **Algèbre d'intervalles d'Allen** — 13 relations, classification, inverse (Partie 2).
3. **Réseau temporel simple (STN)** — matrice des distances, Floyd-Warshall, consistance (Partie 3).
4. **Conflits temporels et scheduling** — exclusion mutuelle, RCPSP-lite glouton, Gantt ASCII (Parties 4-5).

Le notebook Python original atteint ces concepts via **`unified_planning` + `ortools` CP-SAT** ; le twin C# les rend **transparents et exécutables** sans dépendance externe.

### Limitations

- L'algorithme de scheduling est **glouton (LPT)**, non optimal (RCPSP est NP-difficile).
- La table de **composition d'Allen** complète (213 entrées) n'est pas implémentée (exercice).
- La sémantique PDDL 2.1 est **conceptuelle** (pas de solveur complet).

## Exercices


In [7]:
// Exercice 1 : Table de composition d'Allen
// TODO etudiant : implementez la table de composition COMPOSE(R1, R2) retourmant
// l'ENSEMBLE des relations possibles entre I1 et I3 sachant R1(I1,I2) et R2(I2,I3).
// Indice : Allen 1983, Table III. Pour before o before = {before}, etc.
// Etape 1 : representer la table par un dictionnaire (R1,R2) -> HashSet<AllenRel>.
// Etape 2 : remplir au moins before/after/meets/equals (les cas les plus frequents).
public static HashSet<AllenRel> Compose(AllenRel r1, AllenRel r2)
{
    // TODO etudiant
    var result = new HashSet<AllenRel>();
    // Indice : before o before = {before}. Commencez par ce cas.
    return result;
}
Show("Compose(before,before)", string.Join(",", Compose(AllenRel.Before, AllenRel.Before).Select(AllenAlgebra.RelStr)) + " (a completer)");


Compose(before,before):  (a completer)

In [8]:
// Exercice 2 : Deadline dans le STN
// TODO etudiant : ajoutez une contrainte de deadline (t_C <= T_max) au STN1
// et verifiez la consistan ce. Indice : AddConstraint("X0", "C", 0, T_max).
// Etape 1 : choisir un T_max (ex: 12). Etape 2 : re-invoquer FloydWarshall.
public static bool DeadlineConsistent(STN stn, string node, double deadline)
{
    // TODO etudiant
    return true;  // retournez vrai si le STN + deadline reste consistant
}
Show("Deadline a 12 (a completer)", DeadlineConsistent(stn1, "C", 12.0));


Deadline a 12 (a completer): True

In [9]:
// Exercice 3 : Optimalite du makespan (comparaison brute-force)
// TODO etudiant : pour le scenario atelier (5 taches), calculez le makespan OPTIMAL
// par enumeration des permutations valides (precedence-respecting topological orders)
// et comparez avec le makespan glouton ci-dessus.
// Indice : enumerer les ordres topologiques, pour chacun simuler ScheduleAll, garder min.
// Etape 1 : enumerer les ordres topologiques des taches. Etape 2 : evaluer chacun.
public static double OptimalMakespan(List<Task> tasks, double capacity)
{
    // TODO etudiant
    return double.PositiveInfinity;  // retournez le makespan minimal
}
Show("Makespan optimal (a completer)", OptimalMakespan(tasks, 2.0));


Makespan optimal (a completer): ∞

---

*Twin C# du notebook Python `Planners-8-Temporal.ipynb`. Marathon parité .NET ⇄ Python (#4956, Prong B). From-scratch BCL .NET, 0 NuGet, 0 unified_planning / ortools.*
